# ПЗ 3 — OCR субтитров из кадров

Читаем кадры из Drive, вырезаем нижнюю полосу с субтитрами, распознаём текст через EasyOCR.

In [ ]:
!pip install easyocr opencv-python-headless pandas tqdm -q

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive', force_remount=True)

BASE_DRIVE  = '/content/drive/MyDrive/cv-frames'
VIDEO_DIR   = f'{BASE_DRIVE}/видио'
FRAMES_DIR  = f'{BASE_DRIVE}/кадры'
RESULTS_DIR = f'{BASE_DRIVE}/результаты'

for d in [VIDEO_DIR, FRAMES_DIR, RESULTS_DIR]:
    os.makedirs(d, exist_ok=True)
frames = sorted(f for f in os.listdir(FRAMES_DIR) if f.endswith('.jpg'))
print(f'кадров: {len(frames)}')

In [ ]:
import cv2
import easyocr
import pandas as pd
from tqdm.notebook import tqdm

reader = easyocr.Reader(['ru', 'en'], gpu=True)

SUBTITLE_FRACTION = 0.15  # нижние 15% кадра

rows = []

for fname in tqdm(frames, desc='ocr'):
    img = cv2.imread(f'{FRAMES_DIR}/{fname}')
    if img is None:
        continue
    h = img.shape[0]
    crop = img[int(h * (1 - SUBTITLE_FRACTION)):, :]
    gray = cv2.cvtColor(crop, cv2.COLOR_BGR2GRAY)

    for _, text, prob in reader.readtext(gray):
        text = text.strip()
        if text and prob > 0.4:
            rows.append({'frame': fname, 'text': text, 'conf': round(prob, 3)})

df = pd.DataFrame(rows)
df.to_csv(f'{RESULTS_DIR}/ocr_results.csv', index=False, encoding='utf-8-sig')
print(f'распознано строк: {len(df)}')

In [ ]:
unique = df['text'].drop_duplicates().reset_index(drop=True)
print(f'уникальных фраз: {len(unique)}')
print(unique.head(20).to_string())